<a href="https://colab.research.google.com/github/darrickpang/Email/blob/master/Traffic_near_miss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
!pip -q install ultralytics supervision opencv-python-headless numpy

In [9]:
!pip -q install fiftyone ultralytics supervision opencv-python-headless

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 94.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.4/212.4 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.6/112.6 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.4/112.4 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 110.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.1/313.1 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 934.3/934.3 kB 24.6 MB/s eta 0:0

In [20]:
from collections import defaultdict, deque
traj = defaultdict(lambda: deque(maxlen=25))  # last N centers per track

In [10]:
import fiftyone as fo
import fiftyone.zoo as foz

# Download a sample driving video (rainy / low visibility included)
dataset = foz.load_zoo_dataset(
    "quickstart-video",
    max_samples=1
)

sample = dataset.first()
video_path = sample.filepath

video_path

/usr/local/lib/python3.12/dist-packages/glob2/fnmatch.py:141: SyntaxWarning: invalid escape sequence '\Z'
  return '(?ms)' + res + '\Z'


INFO:fiftyone.zoo.datasets:Downloading dataset to '/root/fiftyone/quickstart-video'


INFO:fiftyone.zoo.datasets.base:Downloading dataset...


 100% |████|  281.7Mb/281.7Mb [339.8ms elapsed, 0s remaining, 828.8Mb/s]      


INFO:eta.core.utils: 100% |████|  281.7Mb/281.7Mb [339.8ms elapsed, 0s remaining, 828.8Mb/s]      


...

 100% |█████████████████████| 1/1 [716.4ms elapsed, 0s remaining, 1.4 samples/s] 


INFO:eta.core.utils: 100% |█████████████████████| 1/1 [716.4ms elapsed, 0s remaining, 1.4 samples/s] 


Dataset 'quickstart-video-1' created


INFO:fiftyone.zoo.datasets:Dataset 'quickstart-video-1' created


'/root/fiftyone/quickstart-video/data/Ulcb3AjxM5g_053-1.mp4'

In [11]:
from google.colab import drive
drive.mount("/content/drive")

!mkdir -p /content/drive/MyDrive/driving_video
!cp "{video_path}" /content/drive/MyDrive/driving_video/rain_video.mp4

Mounted at /content/drive


In [12]:
VIDEO_IN  = "/content/drive/MyDrive/driving_video/rain_video.mp4"
VIDEO_OUT = "/content/drive/MyDrive/driving_video/rain_near_miss_output.mp4"

In [14]:
import cv2
import numpy as np
from ultralytics import YOLO
import supervision as sv

# --- Model ---
model = YOLO("yolov8n.pt")  # or yolov8s.pt for better results

# COCO class IDs we care about:
# 0=person, 1=bicycle, 2=car, 3=motorcycle, 5=bus, 7=truck
KEEP_CLASS_IDS = {0, 1, 2, 3, 5, 7}

# --- Video IO ---
cap = cv2.VideoCapture(VIDEO_IN)
if not cap.isOpened():
    raise FileNotFoundError(f"Could not open video: {VIDEO_IN}")

fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(VIDEO_OUT, fourcc, fps, (w, h))

# --- Tracker + Annotators ---
tracker = sv.ByteTrack(frame_rate=fps)

box_annotator   = sv.BoxAnnotator()
label_annotator = sv.LabelAnnotator()

print("FPS:", fps, "Size:", (w, h))

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
FPS: 29.97002997002997 Size: (1920, 1080)


In [19]:
from collections import defaultdict, deque

# Track history per ID
center_hist = defaultdict(lambda: deque(maxlen=10))  # (cx, cy) over last frames
area_hist   = defaultdict(lambda: deque(maxlen=10))  # bbox area proxy for "getting closer"

def compute_near_miss(track_id, cx, cy, area, fps):
    """
    Returns (is_near_miss, score, details_dict) using pixel-space heuristics.
    """
    ch = center_hist[track_id]
    ah = area_hist[track_id]

    ch.append((cx, cy))
    ah.append(area)

    # Need enough history
    if len(ch) < 6:
        return False, 0.0, {}

    # Motion in pixel space
    (x0, y0) = ch[0]
    (x1, y1) = ch[-1]
    dt = (len(ch) - 1) / fps

    vx = (x1 - x0) / dt
    vy = (y1 - y0) / dt

    # "Approaching" proxy:
    # - object moves downward (toward bottom of image) OR
    # - bbox area increases (object getting larger)
    a0, a1 = ah[0], ah[-1]
    da = (a1 - a0) / dt

    approaching = (vy > 40) or (da > 25000)  # tune thresholds per video

    # Risk zone proxy: close to bottom-center region (ego path-ish)
    bottom_zone = (cy > 0.62 * h)
    center_zone = (abs(cx - (w/2)) < 0.22 * w)

    # score (simple)
    score = 0.0
    score += min(max(vy / 300.0, 0), 1.0) * 0.5
    score += min(max(da / 120000.0, 0), 1.0) * 0.5

    is_nm = approaching and bottom_zone and center_zone and score > 0.55

    return is_nm, float(score), {"vy": float(vy), "da": float(da), "cy": float(cy), "cx": float(cx)}

In [21]:
frame_idx = 0
near_miss_events = []  # store (frame_idx, track_id, score, details)

while True:
    ok, frame = cap.read()
    if not ok:
        break
    frame_idx += 1

    # YOLO inference
    results = model(frame, verbose=False)[0]

    # Convert to supervision detections
    detections = sv.Detections.from_ultralytics(results)

    # Filter classes + conf
    if detections.class_id is None or len(detections) == 0:
        writer.write(frame)
        continue

    mask = np.array([(cid in KEEP_CLASS_IDS) for cid in detections.class_id], dtype=bool)
    detections = detections[mask]
    # Optional: confidence filter
    if detections.confidence is not None:
        detections = detections[detections.confidence > 0.35]

    # Track
    tracked = tracker.update_with_detections(detections)

    # Build labels + near-miss marking
    labels = []
    for i in range(len(tracked)):
        x1, y1, x2, y2 = tracked.xyxy[i]
        track_id = int(tracked.tracker_id[i]) if tracked.tracker_id is not None else -1
        cls_id   = int(tracked.class_id[i]) if tracked.class_id is not None else -1
        conf     = float(tracked.confidence[i]) if tracked.confidence is not None else 0.0

        cx = float((x1 + x2) / 2.0)
        cy = float((y1 + y2) / 2.0)
        area = float((x2 - x1) * (y2 - y1))

        is_nm, score, details = compute_near_miss(track_id, cx, cy, area, fps)

        name = model.names.get(cls_id, str(cls_id))
        base = f"ID {track_id} {name} {conf:.2f}"

        if is_nm:
            base += f"  !!! NM score={score:.2f}"
            near_miss_events.append((frame_idx, track_id, score, details))

        labels.append(base)

    # Annotate
    annotated = frame.copy()
    annotated = box_annotator.annotate(annotated, tracked)
    annotated = label_annotator.annotate(annotated, tracked, labels=labels)

    # Add a header overlay
    if len(near_miss_events) > 0 and near_miss_events[-1][0] == frame_idx:
        cv2.putText(annotated, "NEAR-MISS DETECTED", (30, 60),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 0, 255), 3)

    writer.write(annotated)

cap.release()
writer.release()

print("Done. Saved:", VIDEO_OUT)
print("Near-miss events found:", len(near_miss_events))

Done. Saved: /content/drive/MyDrive/driving_video/rain_near_miss_output.mp4
Near-miss events found: 0


In [22]:
for i in range(len(tracked)):
    x1, y1, x2, y2 = tracked.xyxy[i]

    track_id = int(tracked.tracker_id[i]) if tracked.tracker_id is not None else -1
    cls_id   = int(tracked.class_id[i]) if tracked.class_id is not None else -1
    conf     = float(tracked.confidence[i]) if tracked.confidence is not None else 0.0

    cx = float((x1 + x2) / 2.0)
    cy = float((y1 + y2) / 2.0)
    area = float((x2 - x1) * (y2 - y1))

    # ✅ trajectory update MUST be here
    traj[track_id].append((int(cx), int(cy)))

    is_nm, score, details = compute_near_miss(
        track_id, cx, cy, area, fps
    )

    name = model.names.get(cls_id, str(cls_id))
    base = f"ID {track_id} {name} {conf:.2f}"

    if is_nm:
        base += f"  !!! NM score={score:.2f}"

    labels.append(base)

In [23]:
traj[track_id].append((int(cx), int(cy)))